# Nemotron tool-calling agent with `create_agent`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-nvidia/blob/main/cookbook/nemotron_tool_calling_agent.ipynb)

LangChain v1 ships a production-ready agent loop,
[`langchain.agents.create_agent`](https://docs.langchain.com/oss/python/langchain/agents): the model calls tools
until it can answer, with optional memory, middleware, and structured output. This notebook runs it on
`nvidia/nemotron-3-super-120b-a12b` through `ChatNVIDIA`, on NVIDIA-hosted endpoints or a self-hosted NIM.

The tools are local Python functions, so no third-party API keys are needed.

## Install the packages

In [ ]:
%pip install -qU "langchain-nvidia-ai-endpoints>=1.4,<2" "langchain>=1.0,<2" "langgraph>=1.0,<2"

## Configure the endpoint: hosted or self-hosted

Every model client in this notebook is built from the next cell.

| Variable | Unset (default) | Set |
| --- | --- | --- |
| `NVIDIA_BASE_URL` | NVIDIA-hosted endpoints on [build.nvidia.com](https://build.nvidia.com/) | Your own NIM, e.g. `http://localhost:8000/v1` |
| `NVIDIA_API_KEY` | Required for hosted endpoints (you are prompted if it is missing) | Optional for most self-hosted NIMs |
| `NVIDIA_CHAT_MODEL` | `nvidia/nemotron-3-super-120b-a12b` | The model your NIM serves (see `GET $NVIDIA_BASE_URL/models`) |

In [ ]:
import getpass
import os

# Unset: NVIDIA-hosted endpoints (build.nvidia.com).
# Set, e.g. "http://localhost:8000/v1": a self-hosted NIM.
NVIDIA_BASE_URL = os.environ.get("NVIDIA_BASE_URL") or None
CHAT_MODEL = os.environ.get("NVIDIA_CHAT_MODEL", "nvidia/nemotron-3-super-120b-a12b")

if NVIDIA_BASE_URL:
    print(f"Using self-hosted NIM at {NVIDIA_BASE_URL}")
else:
    print("Using NVIDIA-hosted endpoints (build.nvidia.com)")
    if not os.environ.get("NVIDIA_API_KEY"):
        os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA API key (nvapi-...): ")

# Shared keyword arguments for every NVIDIA client in this notebook.
# ChatNVIDIA reads NVIDIA_API_KEY from the environment.
NIM_KWARGS = {"base_url": NVIDIA_BASE_URL} if NVIDIA_BASE_URL else {}
print(f"Chat model: {CHAT_MODEL}")

## Create the chat model

Use `ChatNVIDIA` directly, or LangChain's provider-agnostic
[`init_chat_model`](https://docs.langchain.com/oss/python/langchain/models). Since `langchain` 1.0,
`init_chat_model` maps the `nvidia` provider prefix to `langchain_nvidia_ai_endpoints.ChatNVIDIA`, so both lines
below build the same client.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(model=CHAT_MODEL, temperature=0, **NIM_KWARGS)

same_llm = init_chat_model(f"nvidia:{CHAT_MODEL}", temperature=0, **NIM_KWARGS)
print(type(same_llm).__name__, same_llm.model)

## Define tools

A safe calculator (no `eval`) and a unit converter.

In [ ]:
import ast
import operator

from langchain_core.tools import tool

_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def _evaluate(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPERATORS:
        left, right = _evaluate(node.left), _evaluate(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 100:
            raise ValueError("exponent too large")
        return _OPERATORS[type(node.op)](left, right)
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPERATORS:
        return _OPERATORS[type(node.op)](_evaluate(node.operand))
    raise ValueError("only numbers and + - * / // % ** ( ) are allowed")


@tool
def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression, for example '(12.5 + 7) * 3'."""
    try:
        return str(_evaluate(ast.parse(expression, mode="eval").body))
    except (SyntaxError, ValueError, ZeroDivisionError) as exc:
        return f"Error: {exc}"


calculator.invoke({"expression": "(12.5 + 7) * 3"})

In [ ]:
_TO_BASE = {
    # length (meters)
    "m": ("length", 1.0),
    "km": ("length", 1000.0),
    "mi": ("length", 1609.344),
    "ft": ("length", 0.3048),
    # mass (kilograms)
    "kg": ("mass", 1.0),
    "g": ("mass", 0.001),
    "lb": ("mass", 0.45359237),
}


@tool
def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    """Convert a value between units: m, km, mi, ft (length) or kg, g, lb (mass)."""
    try:
        from_kind, from_factor = _TO_BASE[from_unit.lower()]
        to_kind, to_factor = _TO_BASE[to_unit.lower()]
    except KeyError as exc:
        return f"Error: unknown unit {exc}"
    if from_kind != to_kind:
        return f"Error: cannot convert {from_kind} to {to_kind}"
    return f"{value * from_factor / to_factor:.4g} {to_unit}"


tools = [calculator, convert_units]
convert_units.invoke({"value": 5, "from_unit": "km", "to_unit": "mi"})

## Create and run the agent

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a precise assistant. Use the calculator for arithmetic and "
        "convert_units for unit conversions. Never do math in your head."
    ),
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "I ran 5 km on Monday and 3.5 miles on Tuesday. "
                "How many miles did I run in total?",
            }
        ]
    }
)
for message in result["messages"]:
    message.pretty_print()

`create_agent` also accepts a model string. `create_agent(f"nvidia:{CHAT_MODEL}", tools=tools)` builds the
model with `init_chat_model`, which reads `NVIDIA_BASE_URL` and `NVIDIA_API_KEY` from the environment.

## Stream the steps

`stream_mode="updates"` yields each model and tool step as it happens.

In [ ]:
for update in agent.stream(
    {"messages": [{"role": "user", "content": "What is (17.5 * 4) - 2**5?"}]},
    stream_mode="updates",
):
    for step, data in update.items():
        print(f"--- {step}")
        data["messages"][-1].pretty_print()

## Conversation memory

Add a checkpointer and pass a `thread_id` to keep context across turns.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(model=llm, tools=tools, checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "demo"}}

memory_agent.invoke({"messages": [{"role": "user", "content": "My bag weighs 12 kg."}]}, config)
reply = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "How much is that in pounds?"}]}, config
)
reply["messages"][-1].pretty_print()

## Next steps

- Add planning, a filesystem, and subagents with Deep Agents:
  [nemotron_deep_agent_nim.ipynb](https://github.com/langchain-ai/langchain-nvidia/blob/main/cookbook/nemotron_deep_agent_nim.ipynb).
- Build the same loop by hand with LangGraph:
  [nvidia_nim_agents.ipynb](https://github.com/langchain-ai/langchain-nvidia/blob/main/cookbook/nvidia_nim_agents.ipynb).
- [LangChain agents documentation](https://docs.langchain.com/oss/python/langchain/agents)

## Related

- [Cookbook index](https://github.com/langchain-ai/langchain-nvidia/tree/main/cookbook)
- [`langchain-nvidia-ai-endpoints` README](https://github.com/langchain-ai/langchain-nvidia/blob/main/libs/ai-endpoints/README.md)
- [Model registry: current and deprecated model IDs](https://github.com/langchain-ai/langchain-nvidia/blob/main/libs/ai-endpoints/docs/model_registry.md)
- [NVIDIA NIM for LLMs documentation](https://docs.nvidia.com/nim/large-language-models/latest/introduction.html)